In [7]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

Apply RevIN, preprocess data

In [9]:
df = pd.read_csv("fred.csv")
variable = "INDPRO"   
p = 12
series = df[variable].dropna().to_numpy(dtype=np.float32)

In [10]:
X = []
y = []

for i in range(p, len(series)):
    X.append(series[i-p:i])
    y.append(series[i])

X = torch.tensor(np.array(X)).unsqueeze(2)
y = torch.tensor(np.array(y)).unsqueeze(1)

In [ ]:
n = len(X)

train_end = int(0.7 * n)
val_end = int(0.85 * n)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32,
    shuffle=False
)

In [ ]:
def normalize(x):
    mean = x.mean(dim=1, keepdim=True)
    std = x.std(dim=1, keepdim=True, unbiased=False)

    return (x - mean) / (std + 1e-5), mean, std

def denormalize(x, mean, std):
    return x * (std.squeeze(1) + 1e-5) + mean.squeeze(1)

In [ ]:
class LSTMAttention(nn.Module):
    def __init__(self, input_dim = 1, hidden_dim = 64, num_layers = 1, dropout = 0.0):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.lstm = nn.LSTM(input_size = input_dim, hidden_size = hidden_dim, num_layers = num_layers, batch_first = True, dropout = dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        #x [batch_size, # of lags, input_dim]
        hidden_states, _ = self.lstm(x)

        q = hidden_states[:, -1, :]
        # attention scores is the sum of dot products of queries and keys / sqrt(dimension of hidden state)
        scores = torch.bmm(hidden_states, q.unsqueeze(2)).squeeze(2)
        scores = scores/math.sqrt(self.hidden_dim)

        attention_weights = torch.softmax(scores, dim=1)
        context = torch.bmm(attention_weights.unsqueeze(1), hidden_states).squeeze(1)
        self.fc(context) 

        return prediction, attention_weights    
    

In [ ]:
model = LSTMAttention()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
loss_fn = nn.MSELoss()


for epoch in range(100):
    model.train()
    
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        x_norm, mean, std = normalize(x_batch)
        y_norm = ( y_batch - mean.squeeze(1)) / (std.squeeze(1) + 1e-5)

        pred_norm, _ = model(x_norm)  
        loss = loss_fn(pred_norm, y_norm)

        loss.backward()
        optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(epoch + 1, loss.item())

In [ ]:
model.eval()

with torch.no_grad():
    X_norm, mean, std = normalize(X_test)
    pred_norm, attention = model(X_norm)

    predictions = denormalize(pred_norm, mean, std)
